# Period Prediction — LSTM Training (Google Colab)

Trains an LSTM that predicts **days until the next menstrual period** from the previous 21 days of **34 signals**:
hormones (LH, estrogen), cycle phase, resting heart rate, nightly skin temperature, sleep scores & deep-sleep minutes, respiratory rate, heart-rate zones, activity minutes, stress, and 12 Likert self-report symptoms — merged from every raw mcPHASES CSV into **`master_daily.csv`**.

### How to use
1. Open this notebook on [Google Colab](https://colab.research.google.com/).
2. (Optional) **Runtime → Change runtime type → GPU**.
3. Run cells top-to-bottom. Cell 2 asks for **`master_daily.csv`** (~1.4 MB).
4. The final cell downloads `lstm_best.pt`, `lstm_scaler.npz`, `lstm_metrics.json`.

Training split = **GroupKFold by participant** — metrics reflect generalization to people the model has never seen.

## 1. Install dependencies

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn torch

## 2. Upload the dataset

Upload **`master_daily.csv`** — the merged per-day feature table produced by `preprocess.py`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))
except ImportError:
    CSV_PATH = 'master_daily.csv'
print('using:', CSV_PATH)

## 3. Imports & feature list

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import GroupKFold
import matplotlib.pyplot as plt

# Features fed to the LSTM at each timestep. flow_volume_num / is_bleeding /
# period_start / day_in_cycle are excluded (derived from the target).
FEATURE_COLS = [
    'phase_num',
    'lh', 'estrogen',
    'rhr',
    'temp_nightly_temperature',
    'temp_baseline_relative_nightly_standard_deviation',
    'sleep_overall_score', 'sleep_deep_sleep_in_minutes',
    'sleep_resting_heart_rate', 'sleep_restlessness',
    'resp_full_sleep_breathing_rate',
    'active_sedentary', 'active_lightly', 'active_moderately', 'active_very',
    'fitbit_stress_score',
    'hrz_in_default_zone_1', 'hrz_in_default_zone_2', 'hrz_in_default_zone_3',
    'appetite_num', 'exerciselevel_num', 'headaches_num', 'cramps_num',
    'sorebreasts_num', 'fatigue_num', 'sleepissue_num', 'moodswing_num',
    'stress_num', 'foodcravings_num', 'indigestion_num', 'bloating_num',
    'is_weekend_num', 'age_at_interval', 'age_of_first_menarche',
]

WINDOW_SIZE = 21
MAX_HORIZON = 45

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 4. Load the master dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
df['is_weekend_num'] = df['is_weekend'].astype(int)
df = df.sort_values(['id', 'study_interval', 'day_in_study']).reset_index(drop=True)
print(f'rows: {len(df):,}  cols: {df.shape[1]}  participants: {df["id"].nunique()}')
print(f'period_starts detected: {int(df["period_start"].sum())}')
print(f'target (days_to_next_period) summary:')
print(df['days_to_next_period'].describe().round(2))

## 5. Build sliding-window sequences

In [ ]:
def ffill(a):
    out = a.copy()
    for j in range(out.shape[1]):
        col = out[:, j]; last = np.nan
        for i in range(len(col)):
            if np.isnan(col[i]): col[i] = last
            else: last = col[i]
    return out

def build_windows(df):
    features = [c for c in FEATURE_COLS if c in df.columns]
    X_list, y_list, g_list = [], [], []
    for (pid, _), g in df.groupby(['id', 'study_interval']):
        g = g.sort_values('day_in_study').reset_index(drop=True)
        feat = ffill(g[features].to_numpy(dtype=np.float32))
        feat = np.nan_to_num(feat, nan=0.0)
        tgt = g['days_to_next_period'].to_numpy(dtype=np.float32)
        for end in range(WINDOW_SIZE - 1, len(g)):
            y = tgt[end]
            if np.isnan(y) or y > MAX_HORIZON: continue
            X_list.append(feat[end - WINDOW_SIZE + 1: end + 1])
            y_list.append(y); g_list.append(pid)
    return (np.stack(X_list).astype(np.float32),
            np.asarray(y_list, dtype=np.float32),
            np.asarray(g_list, dtype=np.int64),
            features)

X, y, groups, features = build_windows(df)
print(f'windows: {len(X):,}   shape: {X.shape}   features used: {len(features)}')
print(f'baseline MAE (predict mean): {np.mean(np.abs(y - y.mean())):.2f} days')

## 6. LSTM model

In [ ]:
class PeriodLSTM(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

class WindowDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def train_fold(X_tr, y_tr, X_va, y_va, epochs=60, batch=64, patience=10):
    model = PeriodLSTM(X_tr.shape[2]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.L1Loss()
    tr = DataLoader(WindowDS(X_tr, y_tr), batch_size=batch, shuffle=True)
    va = DataLoader(WindowDS(X_va, y_va), batch_size=batch)
    best, bad, history, best_state = float('inf'), 0, [], None
    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * xb.size(0)
        tl /= len(y_tr)
        model.eval(); preds, tgts = [], []
        with torch.no_grad():
            for xb, yb in va:
                preds.append(model(xb.to(device)).cpu().numpy()); tgts.append(yb.numpy())
        vl = float(np.mean(np.abs(np.concatenate(preds) - np.concatenate(tgts))))
        history.append({'epoch': ep, 'train_mae': tl, 'val_mae': vl})
        if vl < best - 1e-4:
            best, bad = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model, best, history

## 7. Train with GroupKFold (by participant)

No participant appears in both train & validation sets — metrics reflect generalization to unseen people.

In [ ]:
N_FOLDS, EPOCHS, BATCH = 5, 60, 64

gkf = GroupKFold(n_splits=N_FOLDS)
results, best_overall, best_state, best_scaler, best_hist = [], float('inf'), None, None, None

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    X_tr, X_va, y_tr, y_va = X[tr_idx], X[va_idx], y[tr_idx], y[va_idx]
    flat = X_tr.reshape(-1, X_tr.shape[-1])
    mean, std = flat.mean(axis=0), flat.std(axis=0) + 1e-6
    X_tr, X_va = (X_tr - mean) / std, (X_va - mean) / std
    model, mae, hist = train_fold(X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH)
    print(f'[fold {fold}]  val MAE = {mae:.2f} days  (train {len(tr_idx)} / val {len(va_idx)})')
    results.append({'fold': fold, 'val_mae': mae, 'n_train': int(len(tr_idx)), 'n_val': int(len(va_idx))})
    if mae < best_overall:
        best_overall, best_state, best_scaler, best_hist = mae, {k: v.clone() for k, v in model.state_dict().items()}, (mean, std), hist

cv_mae = float(np.mean([r['val_mae'] for r in results]))
print(f'\nCV mean val MAE: {cv_mae:.2f} days   best fold: {best_overall:.2f} days')

## 8. Training curve for the best fold

In [ ]:
h = pd.DataFrame(best_hist)
plt.figure(figsize=(8, 4))
plt.plot(h['epoch'], h['train_mae'], label='train MAE')
plt.plot(h['epoch'], h['val_mae'], label='val MAE')
plt.xlabel('epoch'); plt.ylabel('MAE (days)')
plt.title('Best-fold training curve')
plt.grid(alpha=0.3); plt.legend(); plt.show()

## 9. Save artefacts & download

In [ ]:
torch.save(best_state, 'lstm_best.pt')
np.savez('lstm_scaler.npz', mean=best_scaler[0], std=best_scaler[1],
         feature_cols=np.array(features))
metrics = {
    'device': device, 'window_size': WINDOW_SIZE,
    'n_features': int(X.shape[2]), 'n_windows': int(len(X)),
    'feature_cols': features,
    'fold_results': results,
    'cv_mean_val_mae': cv_mae, 'best_val_mae': best_overall,
    'baseline_mae_mean': float(np.mean(np.abs(y - y.mean()))),
}
with open('lstm_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved lstm_best.pt, lstm_scaler.npz, lstm_metrics.json')

try:
    from google.colab import files
    files.download('lstm_best.pt')
    files.download('lstm_scaler.npz')
    files.download('lstm_metrics.json')
except ImportError:
    print('Local run — artefacts saved in the working directory.')

## 10. (Optional) Inference on a participant's last 21 days

In [ ]:
def predict_days_to_next_period(recent_days_df, model, mean, std, features):
    assert len(recent_days_df) >= WINDOW_SIZE, f'need >= {WINDOW_SIZE} days'
    d = recent_days_df.copy()
    if 'is_weekend_num' not in d.columns and 'is_weekend' in d.columns:
        d['is_weekend_num'] = d['is_weekend'].astype(int)
    x = d[features].tail(WINDOW_SIZE).to_numpy(dtype=np.float32)
    x = ffill(x); x = np.nan_to_num(x, nan=0.0); x = (x - mean) / std
    model.eval()
    with torch.no_grad():
        return float(model(torch.from_numpy(x[None, ...]).to(device)).item())

demo_model = PeriodLSTM(X.shape[2]).to(device)
demo_model.load_state_dict(best_state)
pid = df['id'].iloc[0]
recent = df[df['id'] == pid].sort_values('day_in_study').tail(WINDOW_SIZE)
days = predict_days_to_next_period(recent, demo_model, *best_scaler, features)
print(f'Participant {pid} — predicted days until next period: {days:.1f}')